<a href="https://colab.research.google.com/github/rayaguilos06/flyrank-ml-internship/blob/main/w05_model-Aguilos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
TYPE huggingface,
TOKEN '{HF_TOKEN}'
)
""")

TABLES = {
    "fact_daily":
    "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/**/*.parquet')"
}

print("Connection Successful")

Connection Successful


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# ==========================================================
# SECTION 1 — METHOD CHOICE AND WHY
#
# Modeling Method:
# Random Forest Classifier
#
# Reason for choosing this method:
# - Works well on structured/tabular datasets.
# - Handles nonlinear relationships between features.
# - Less prone to overfitting than a single Decision Tree.
# - Provides feature importance for interpretation.
# - Appropriate for comparing against my Week 4 baseline.
# ==========================================================

import pandas as pd

print("Loading a sample from the warehouse...")

df = con.sql(f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    gsc_impressions,
    gsc_clicks,
    gsc_avg_position,
    ga4_pageviews,
    ga4_sessions,
    ga4_users,
    sessions_organic,
    scroll_events
FROM {TABLES["fact_daily"]}
WHERE month='2025-01'
LIMIT 50000
""").df()

print("Sample Loaded Successfully")
print()

print("Dataset Shape:")
print(df.shape)

print()

print("Columns:")
print(df.columns.tolist())

print()

print("First Five Rows:")
display(df.head())

Loading a sample from the warehouse...
Sample Loaded Successfully

Dataset Shape:
(1297, 11)

Columns:
['report_date', 'client_hash_id', 'content_hash_id', 'gsc_impressions', 'gsc_clicks', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'sessions_organic', 'scroll_events']

First Five Rows:


,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions,ga4_users,sessions_organic,scroll_events
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,30,0,3.833333,0,0,0,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,5,0,71.600000,0,0,0,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,1,0,34.000000,0,0,0,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,6,0,23.333333,0,0,0,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,5,0,17.800000,0,0,0,0,0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# ============================================
# SECTION 2
# Split Design
# ============================================

from sklearn.model_selection import train_test_split

print("=" * 60)
print("Split Design")
print("=" * 60)

print("""
Method:
• Random Train/Test Split
• 80% Training
• 20% Testing

Reason:
This assignment compares a machine learning model
against the Week 4 baseline using the same data.
A simple random split provides an honest evaluation
while preventing the model from being tested on
the same rows it was trained on.
""")

# ---------------------------------------
# Create Target Variable
# ---------------------------------------

# Target:
# 1 = Page received at least one click
# 0 = Page received no clicks

df["target"] = (df["gsc_clicks"] > 0).astype(int)

# ---------------------------------------
# Selected Features
# ---------------------------------------

features = [
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "sessions_organic",
    "scroll_events"
]

X = df[features]
y = df["target"]

# ---------------------------------------
# Train/Test Split
# ---------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining Rows :", X_train.shape[0])
print("Testing Rows  :", X_test.shape[0])

print("\nTarget Distribution")
print(y.value_counts())

print("\nTraining Feature Shape:", X_train.shape)
print("Testing Feature Shape :", X_test.shape)

Split Design

Method:
• Random Train/Test Split
• 80% Training
• 20% Testing

Reason:
This assignment compares a machine learning model
against the Week 4 baseline using the same data.
A simple random split provides an honest evaluation
while preventing the model from being tested on
the same rows it was trained on.


Training Rows : 1037
Testing Rows  : 260

Target Distribution
target
0    1220
1      77
Name: count, dtype: int64

Training Feature Shape: (1037, 7)
Testing Feature Shape : (260, 7)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# ==========================================================
# SECTION 3
# Train + Compare vs My Baseline
# ==========================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("="*60)
print("Training Random Forest Model")
print("="*60)

# -----------------------------------------
# Train Model
# -----------------------------------------

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

# -----------------------------------------
# Predictions
# -----------------------------------------

predictions = model.predict(X_test)

# -----------------------------------------
# Model Metrics
# -----------------------------------------

model_accuracy = accuracy_score(y_test, predictions)
model_precision = precision_score(y_test, predictions, zero_division=0)
model_recall = recall_score(y_test, predictions, zero_division=0)
model_f1 = f1_score(y_test, predictions, zero_division=0)

# -----------------------------------------
# Week 4 Baseline
# (Predict every page has NO action)
# -----------------------------------------

baseline_predictions = [0] * len(y_test)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)
baseline_precision = precision_score(
    y_test,
    baseline_predictions,
    zero_division=0
)

baseline_recall = recall_score(
    y_test,
    baseline_predictions,
    zero_division=0
)

baseline_f1 = f1_score(
    y_test,
    baseline_predictions,
    zero_division=0
)

# -----------------------------------------
# Comparison Table
# -----------------------------------------

comparison = pd.DataFrame({
    "Model":[
        "Week 4 Baseline",
        "Random Forest"
    ],
    "Accuracy":[
        baseline_accuracy,
        model_accuracy
    ],
    "Precision":[
        baseline_precision,
        model_precision
    ],
    "Recall":[
        baseline_recall,
        model_recall
    ],
    "F1 Score":[
        baseline_f1,
        model_f1
    ]
})

print("\nModel Comparison")
display(comparison)

Training Random Forest Model

Model Comparison


,Model,Accuracy,Precision,Recall,F1 Score
0,Week 4 Baseline,0.942308,0.000000,0.000000,0.000000
1,Random Forest,0.934615,0.333333,0.133333,0.190476


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# ==========================================================
# SECTION 4
# Errors and Interpretation
# ==========================================================

from sklearn.metrics import confusion_matrix
import pandas as pd

print("=" * 60)
print("Errors and Interpretation")
print("=" * 60)

# ---------------------------------------
# Confusion Matrix
# ---------------------------------------

cm = confusion_matrix(y_test, predictions)

cm_df = pd.DataFrame(
    cm,
    index=["Actual No Click", "Actual Click"],
    columns=["Predicted No Click", "Predicted Click"]
)

print("\nConfusion Matrix")
display(cm_df)

# ---------------------------------------
# Feature Importance
# ---------------------------------------

importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nFeature Importance")
display(importance)

# ---------------------------------------
# Interpretation
# ---------------------------------------

print("\nInterpretation")
print("-" * 60)

print("""
Observed findings:

• The Random Forest model successfully identified some
  pages that were likely to receive clicks.

• The baseline achieved higher overall accuracy because
  most pages had zero clicks, making the dataset
  highly imbalanced.

• The Random Forest produced non-zero Precision,
  Recall, and F1 Score, showing that it learned
  meaningful patterns instead of always predicting
  the majority class.

• Features with higher importance contributed more
  to the model's decisions and may be useful for
  future feature engineering.

Overall, this model should be treated as
decision-support rather than an automated
decision-making system.
""")

Errors and Interpretation

Confusion Matrix


,Predicted No Click,Predicted Click
Actual No Click,241,4
Actual Click,13,2



Feature Importance


,Feature,Importance
1,gsc_avg_position,0.64422
0,gsc_impressions,0.35578
2,ga4_pageviews,0.00000
3,ga4_sessions,0.00000
4,ga4_users,0.00000
5,sessions_organic,0.00000
6,scroll_events,0.00000



Interpretation
------------------------------------------------------------

Observed findings:

• The Random Forest model successfully identified some
  pages that were likely to receive clicks.

• The baseline achieved higher overall accuracy because
  most pages had zero clicks, making the dataset
  highly imbalanced.

• The Random Forest produced non-zero Precision,
  Recall, and F1 Score, showing that it learned
  meaningful patterns instead of always predicting
  the majority class.

• Features with higher importance contributed more
  to the model's decisions and may be useful for
  future feature engineering.

Overall, this model should be treated as
decision-support rather than an automated
decision-making system.



## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.